# ⚙️ BPS Installation — MAAP Coding Environment ⚙️

**Procedure for installing BPS within the MAAP Coding/Experiment environment.**

This notebook guides through the full installation of the BPS Processor Suite (BPS) on the MAAP Coding environment, starting from the raw tarballs up to the verification of all processors.

---

## 📋 Overview

| Step | Description |
|---|---|
| **0. Configuration** | Define BPS version and all relevant paths |
| **1. Extract tarballs** | Untar BPS bundle and BTK into the correct directories |
| **2. Conda environment** | Create and register the BPS conda environment |
| **3. Install BPS packages** | Build local conda channel and install all BPS processors |
| **4. Patch `set_environment.bash`** | Update environment script with correct paths and add shell alias |
| **5. Verify installation** | Run `--help` on all processors to confirm successful installation |

---

> ⚠️ **Prerequisites**
> - Tarballs `bps-bundle-v{ver}.tar.gz` and `btk-vt-{ver}.tar.gz` must be available in `/home/jovyan/SW/BPS_V{ver}/`

## 0. ⚙️ Configuration

Define all the path variables used throughout the notebook.  
Edit `BPS_VERSION` here to target a different BPS release — all paths are derived automatically.

> 📂 **Expected folder structure before starting:**
> ```
> /home/jovyan/SW/BPS_V450/
> ├── bps-bundle-v4.5.0.tar.gz
> ├── btk-vt-4.5.0.tar.gz
> └── ...
> ```

| Variable | Description |
|---|---|
| `BPS_VERSION` | BPS version to install (e.g. `4.5.0`) |
| `env_DIR` | Conda environments directory (`/home/jovyan/env`) |
| `SW_DIR` | Directory containing the BPS version folders (`/home/jovyan/SW`) |
| `DATA_DIR` | Data directory (`/home/jovyan/DATA`) |
| `CONDA_ENV` | Conda environment name (e.g. `BPS_450`) |
| `CONFIGURATION_BPS_DIR` | BTK deployment directory (e.g. `/home/jovyan/BPS/BPS_V450`) |

In [1]:
HOME           = "/home/jovyan"
BPS_VERSION    = "4.4.2"
env_DIR    = f"{HOME}/env"    #/home/jovyan/env
SW_DIR     = f"{HOME}/SW"
DATA_DIR  = f"{HOME}/DATA"
_ver_compact   = BPS_VERSION.replace(".", "")
CONDA_ENV      = f"BPS_{_ver_compact}"
CONFIGURATION_BPS_DIR        = f"{HOME}/BPS/BPS_V{_ver_compact}"




print("Configuration:")
print(f"  HOME : {HOME}")
print(f"  BPS_VERSION: {BPS_VERSION}")
print(f"  env_DIR    : {env_DIR}")
print(f"  SW_DIR     : {SW_DIR}")
print(f"  DATA_DIR   : {DATA_DIR}")
print(f"  CONDA_ENV  : {CONDA_ENV}")
print(f"  CONFIGURATION_BPS_DIR  : {CONFIGURATION_BPS_DIR}")


Configuration:
  HOME : /home/jovyan
  BPS_VERSION: 4.4.2
  env_DIR    : /home/jovyan/env
  SW_DIR     : /home/jovyan/SW
  DATA_DIR   : /home/jovyan/DATA
  CONDA_ENV  : BPS_442
  CONFIGURATION_BPS_DIR  : /home/jovyan/BPS/BPS_V442


## 1. 📦 Extract tarballs

Extracts the BPS bundle and BTK tarballs from `SW_DIR/BPS_V{ver}/`.

- **`bps-bundle-v*.tar.gz`** → extracted in-place inside `SW_DIR/BPS_V{ver}/` (→ `bundle/`)
- **`btk-vt-*.tar.gz`** → extracted in-place, then its content is copied to `CONFIGURATION_BPS_DIR`

> ℹ️ If `bundle/` or the BTK folder already exist, extraction is skipped.

In [2]:
%%bash -s "$CONFIGURATION_BPS_DIR" "$BPS_VERSION" "$SW_DIR"
CONFIGURATION_BPS_DIR=$1
BPS_VERSION=$2
SW_DIR=$3

BPS_SW_DIR="${SW_DIR}/BPS_V${BPS_VERSION//./}"   # es. /home/jovyan/SW/BPS_V442
BTK_DIR="${BPS_SW_DIR}/btk-vt-${BPS_VERSION}"    # es. /home/jovyan/SW/BPS_V442/btk-vt-4.4.2

cd "${BPS_SW_DIR}"

# Extract bps-bundle in-place
if [ -d "bundle" ]; then
    echo "ℹ️  bundle/ already extracted, skipping"
else
    tar -xzf "bps-bundle-v${BPS_VERSION}.tar.gz"
    echo "✅ bps-bundle extracted in ${BPS_SW_DIR}"
fi

# Extract btk in-place
if [ -d "${BTK_DIR}" ]; then
    echo "ℹ️  ${BTK_DIR} already extracted, skipping"
else
    tar -xzf "btk-vt-${BPS_VERSION}.tar.gz"
    echo "✅ btk extracted in ${BPS_SW_DIR}"
fi

# Copy btk content to CONFIGURATION_BPS_DIR
mkdir -p "${CONFIGURATION_BPS_DIR}"
cp -r "${BTK_DIR}/." "${CONFIGURATION_BPS_DIR}/"
echo "✅ btk content copied to ${CONFIGURATION_BPS_DIR}"

echo ""
echo "=== ${CONFIGURATION_BPS_DIR} ==="
ls "${CONFIGURATION_BPS_DIR}"

ℹ️  bundle/ already extracted, skipping
ℹ️  /home/jovyan/SW/BPS_V442/btk-vt-4.4.2 already extracted, skipping
✅ btk content copied to /home/jovyan/BPS/BPS_V442

=== /home/jovyan/BPS/BPS_V442 ===
configuration_files
internal_resources
pkgs
test_cases
tools


## 2. 🐍 Create Conda environment

Creates the conda environment `CONDA_ENV` (e.g. `BPS_450`) with Python 3.12.

- Creates `env_DIR` (`/home/jovyan/env`) if it does not exist
- Registers `env_DIR` as a conda `envs_dirs`
- Skips creation if the environment already exists

In [3]:
%%bash -s "$CONDA_ENV" "$env_DIR"
CONDA_ENV=$1
env_DIR=$2

# Create envs_dirs folder if it doesn't exist
if [ ! -d "${env_DIR}" ]; then
    mkdir -p "${env_DIR}"
    echo "✅ Folder ${env_DIR} created"
else
    echo "ℹ️  Folder ${env_DIR} already exists"
fi

# Register envs_dirs in conda config
conda config --add envs_dirs "${env_DIR}"

# Create new environment only if it doesn't exist
if conda env list | grep -qE "(^|[[:space:]])${CONDA_ENV}([[:space:]]|$)"; then
    echo "ℹ️  Environment ${CONDA_ENV} already exists, skipping creation"
else
    conda create --name "${CONDA_ENV}" python=3.12 -y
    echo "✅ Environment ${CONDA_ENV} created"
fi

# Final check
echo ""
echo "=== Conda environments ==="
conda env list

ℹ️  Folder /home/jovyan/env already exists


Channels:
 - conda-forge
 - fastai
 - file:///home/jovyan/SW/BPS_V441/bundle/bps_conda_channel
Platform: linux-64
Solving environment: ...working... done




==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda





## Package Plan ##

  environment location: /home/jovyan/env/BPS_442

  added / updated specs:
    - python=3.12


The following NEW packages will be INSTALLED:

  _openmp_mutex      conda-forge/linux-64::_openmp_mutex-4.5-20_gnu 
  bzip2              conda-forge/linux-64::bzip2-1.0.8-hda65f42_9 
  ca-certificates    conda-forge/noarch::ca-certificates-2026.2.25-hbd8a1cb_0 
  icu                conda-forge/linux-64::icu-78.3-h33c6efd_0 
  ld_impl_linux-64   conda-forge/linux-64::ld_impl_linux-64-2.45.1-default_hbd61a6d_102 
  libexpat           conda-forge/linux-64::libexpat-2.7.5-hecca717_0 
  libffi             conda-forge/linux-64::libffi-3.5.2-h3435931_0 
  libgcc             conda-forge/linux-64::libgcc-15.2.0-he0feb66_18 
  libgcc-ng          conda-forge/linux-64::libgcc-ng-15.2.0-h69a702a_18 
  libgomp            conda-forge/linux-64::libgomp-15.2.0-he0feb66_18 
  liblzma            conda-forge/linux-64::liblzma-5.8.3-hb03c661_0 
  libnsl             conda-forge/linux-64::libns

## 3. 🔧 Install BPS packages

Runs the BPS installation inside the conda environment. Equivalent to `install.sh`.

- **Step 1** — installs `conda-build` with `libmamba` solver
- **Step 2** — collects all `.tar.bz2` / `*conda` files and builds a local conda channel (`bps_conda_channel`)
- **Step 3** — registers `fastai`, `conda-forge`, and the local BPS channel
- **Step 4** — installs all BPS processors and dependencies:
  `bps-l1_processor`, `bps-stack_processor`, `bps-l2a_processor`, `bps-l2b_fd_processor`, `bps-l2b_fh_processor`, `bps-l1_framing_processor`, `bps-l2b_agb_processor`, `arepytools`, `arepyextras-quality`

In [4]:
%%bash -s "$CONDA_ENV" "$SW_DIR" "$BPS_VERSION"
CONDA_ENV=$1
SW_DIR=$2
BPS_VERSION=$3

BUNDLE_DIR="${SW_DIR}/BPS_V${BPS_VERSION//./}/bundle"
cd "${BUNDLE_DIR}"

echo "=== Step 1: install conda-build ==="
conda run --name "${CONDA_ENV}" conda install -y conda-build --solver=libmamba

echo "=== Step 2: create BPS conda channel ==="
BPS_CONDA_CHANNEL="${BUNDLE_DIR}/bps_conda_channel"
mkdir -p "${BPS_CONDA_CHANNEL}/noarch"
find . -type f -name "*.tar.bz2" -exec cp {} "${BPS_CONDA_CHANNEL}/noarch" \;
find . -type f -name "*conda"    -exec cp {} "${BPS_CONDA_CHANNEL}/noarch" \;
conda run --name "${CONDA_ENV}" python -m conda_index --verbose "${BPS_CONDA_CHANNEL}"

echo "=== Step 3: add conda channels ==="
conda run --name "${CONDA_ENV}" conda config --prepend channels fastai
conda run --name "${CONDA_ENV}" conda config --prepend channels conda-forge
conda run --name "${CONDA_ENV}" conda config --prepend channels "${BPS_CONDA_CHANNEL}"

echo "=== Step 4: install BPS packages ==="
conda run --name "${CONDA_ENV}" conda install -y \
    bps-l1_processor \
    bps-stack_processor \
    bps-l2a_processor \
    bps-l2b_fd_processor \
    bps-l2b_fh_processor \
    bps-l1_framing_processor \
    bps-l2b_agb_processor \
    arepytools \
    arepyextras-quality \
    --solver=libmamba

echo "✅ BPS packages installed"

=== Step 1: install conda-build ===




==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda





Channels:
 - conda-forge
 - fastai
 - file:///home/jovyan/SW/BPS_V441/bundle/bps_conda_channel
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /home/jovyan/env/BPS_442

  added / updated specs:
    - conda-build


The following NEW packages will be INSTALLED:

  archspec           conda-forge/noarch::archspec-0.2.5-pyhd8ed1ab_0 
  attrs              conda-forge/noarch::attrs-26.1.0-pyhcf101f3_0 
  backports.zstd     conda-forge/linux-64::backports.zstd-1.3.0-py312h90b7ffd_0 
  beautifulsoup4     conda-forge/noarch::beautifulsoup4-4.14.3-pyha770c72_0 
  boltons            conda-forge/noarch::boltons-25.0.0-pyhd8ed1ab_0 
  brotli-python      conda-forge/linux-64::brotli-python-1.2.0-py312hdb49522_1 
  c-ares             conda-forge/linux-64::c-ares-1.34.6-hb03c661_0 
  certifi            conda-forge/noarch::certifi-2026.2.25-pyhd8ed1ab_0 
  cffi               conda-forge/linux-64::cffi-2.0.0-py312h460c074_1 
  chardet            con

cp: './bps_conda_channel/noarch/arepyextras-runner-1.0.2-py_0.tar.bz2' and '/home/jovyan/SW/BPS_V442/bundle/bps_conda_channel/noarch/arepyextras-runner-1.0.2-py_0.tar.bz2' are the same file
cp: './bps_conda_channel/noarch/arepyextras-quality-1.2.8-py_0.tar.bz2' and '/home/jovyan/SW/BPS_V442/bundle/bps_conda_channel/noarch/arepyextras-quality-1.2.8-py_0.tar.bz2' are the same file
cp: './bps_conda_channel/noarch/arepyextras-copernicus_dem_extractor-3.1.4-py_0.tar.bz2' and '/home/jovyan/SW/BPS_V442/bundle/bps_conda_channel/noarch/arepyextras-copernicus_dem_extractor-3.1.4-py_0.tar.bz2' are the same file
cp: './bps_conda_channel/noarch/bps-stack_pre_processor-4.4.2-py_0.conda' and '/home/jovyan/SW/BPS_V442/bundle/bps_conda_channel/noarch/bps-stack_pre_processor-4.4.2-py_0.conda' are the same file
cp: './bps_conda_channel/noarch/bps-l1_core_processor-4.4.2-py_0.conda' and '/home/jovyan/SW/BPS_V442/bundle/bps_conda_channel/noarch/bps-l1_core_processor-4.4.2-py_0.conda' are the same file
cp: 

=== Step 3: add conda channels ===


=== Step 4: install BPS packages ===
Channels:
 - file:///home/jovyan/SW/BPS_V442/bundle/bps_conda_channel
 - conda-forge
 - fastai
 - file:///home/jovyan/SW/BPS_V441/bundle/bps_conda_channel
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /home/jovyan/env/BPS_442

  added / updated specs:
    - arepyextras-quality
    - arepytools
    - bps-l1_framing_processor
    - bps-l1_processor
    - bps-l2a_processor
    - bps-l2b_agb_processor
    - bps-l2b_fd_processor
    - bps-l2b_fh_processor
    - bps-stack_processor


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    arepyextras-quality-1.2.8  |             py_0          66 KB  file:///home/jovyan/SW/BPS_V442/bundle/bps_conda_channel
    arepyextras-runner-1.0.2   |             py_0          10 KB  file:///home/jovyan/SW/BPS_V442/bundle/bps_conda_channel
    arepytools-1.8.1           |



==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda






✅ BPS packages installed


## 4. 📝 Patch `set_environment.bash`

Updates the BPS environment configuration script with the correct paths for the current installation.

- Creates a timestamped backup (`set_environment.bash.bak_YYYYMMDD_HHMMSS`) before any modification
- Patches the following variables via `sed`:

| Variable | New value |
|---|---|
| `BPS_PYTHON_ENV` | `CONDA_ENV` (e.g. `BPS_450`) |
| `BPS_TEST_PLAN_PATH` | `SW_DIR/BPS_V{ver}` |
| `BPS_BUNDLE_DIR` | `SW_DIR/BPS_V{ver}/bundle` |
| `BPS_CONDA_CHANNEL` | `SW_DIR/BPS_V{ver}/bundle/bps_conda_channel` |
| `BPS_TDS_PATH` | `CONFIGURATION_BPS_DIR` |

- Adds a shell alias to `~/.bashrc` (e.g. `bps450`) to source the environment quickly
- Shows a `diff` of the changes applied

In [5]:
%%bash -s "$CONDA_ENV" "$SW_DIR" "$CONFIGURATION_BPS_DIR" "$BPS_VERSION"
CONDA_ENV=$1
SW_DIR=$2
CONFIGURATION_BPS_DIR=$3
BPS_VERSION=$4

_ver_compact="${BPS_VERSION//./}"
SCRIPTS_DIR="${CONFIGURATION_BPS_DIR}/tools/scripts"
SET_ENV="${SCRIPTS_DIR}/set_environment.bash"
BPS_SW_VERSION_DIR="${SW_DIR}/BPS_V${_ver_compact}"
TIMESTAMP=$(date +"%Y%m%d_%H%M%S")

# Go to scripts folder
cd "${SCRIPTS_DIR}"
echo "📂 Working in: $(pwd)"

# Backup with timestamp
BACKUP="${SET_ENV}.bak_${TIMESTAMP}"
cp "${SET_ENV}" "${BACKUP}"
echo "ℹ️  Backup saved: ${BACKUP}"

# Patch only the changed variables
sed -i "s|BPS_PYTHON_ENV=.*|BPS_PYTHON_ENV=${CONDA_ENV}|" "${SET_ENV}"
sed -i "s|BPS_TEST_PLAN_PATH=.*|BPS_TEST_PLAN_PATH=${BPS_SW_VERSION_DIR}|" "${SET_ENV}"
sed -i "s|BPS_BUNDLE_DIR=.*|BPS_BUNDLE_DIR=${BPS_SW_VERSION_DIR}/bundle|" "${SET_ENV}"
sed -i "s|BPS_CONDA_CHANNEL=.*|BPS_CONDA_CHANNEL=${BPS_SW_VERSION_DIR}/bundle/bps_conda_channel|" "${SET_ENV}"
sed -i "s|BPS_TDS_PATH=.*|BPS_TDS_PATH=${CONFIGURATION_BPS_DIR}|" "${SET_ENV}"

echo "✅ set_environment.bash patched"
echo ""
echo "=== diff vs backup ==="
diff "${BACKUP}" "${SET_ENV}" || true

# Add alias to .bashrc (only if not already present)
ALIAS="alias bps${_ver_compact}='source ${SCRIPTS_DIR}/set_environment.bash'"
if grep -qF "${ALIAS}" ~/.bashrc; then
    echo "ℹ️  Alias already present in .bashrc, skipping"
else
    echo "${ALIAS}" >> ~/.bashrc
    echo "✅ Alias added to ~/.bashrc: ${ALIAS}"
fi

📂 Working in: /home/jovyan/BPS/BPS_V442/tools/scripts
ℹ️  Backup saved: /home/jovyan/BPS/BPS_V442/tools/scripts/set_environment.bash.bak_20260414_150333
✅ set_environment.bash patched

=== diff vs backup ===
6c6
< BPS_PYTHON_ENV=bps-dev-env
---
> BPS_PYTHON_ENV=BPS_442
8c8
< BPS_TEST_PLAN_PATH=/storage/projects/biomass_bps/test_plan
---
> BPS_TEST_PLAN_PATH=/home/jovyan/SW/BPS_V442
10,11c10,11
< BPS_BUNDLE_DIR=${BPS_TEST_PLAN_PATH}/tools/install_dir/bundle
< BPS_CONDA_CHANNEL=${BPS_TEST_PLAN_PATH}/tools/install_dir/bps_conda_channel
---
> BPS_BUNDLE_DIR=/home/jovyan/SW/BPS_V442/bundle
> BPS_CONDA_CHANNEL=/home/jovyan/SW/BPS_V442/bundle/bps_conda_channel
13c13
< BPS_TDS_PATH=${BPS_TEST_PLAN_PATH}/tds
---
> BPS_TDS_PATH=/home/jovyan/BPS/BPS_V442
✅ Alias added to ~/.bashrc: alias bps442='source /home/jovyan/BPS/BPS_V442/tools/scripts/set_environment.bash'


## 5. ✅ Verify installation

Sources `set_environment.bash` and runs `--help` on all BPS processors to verify the installation.

Processors tested:
- `bps_l1_processor`
- `bps_l1_framing_processor`
- `bps_stack_processor`
- `bps_l2a_processor`
- `bps_l2b_fd_processor`
- `bps_l2b_fh_processor`
- `bps_l2b_agb_processor`

> ✅ = processor found and responding correctly  
> ❌ = processor missing or installation failed

In [6]:
%%bash -s "$CONDA_ENV" "$CONFIGURATION_BPS_DIR" "$BPS_VERSION"
CONDA_ENV=$1
CONFIGURATION_BPS_DIR=$2
BPS_VERSION=$3

_ver_compact="${BPS_VERSION//./}"
SCRIPTS_DIR="${CONFIGURATION_BPS_DIR}/tools/scripts"

# Source set_environment.bash (equivalent of bps450)
echo "=== Sourcing set_environment.bash ==="
source "${SCRIPTS_DIR}/set_environment.bash"
echo "✅ Environment activated"
echo ""

# Test all processors
PROCESSORS=(
    "bps_l1_processor"
    "bps_l1_framing_processor"
    "bps_stack_processor"
    "bps_l2a_processor"
    "bps_l2b_fd_processor"
    "bps_l2b_fh_processor"
    "bps_l2b_agb_processor"
)

echo "=== Processor --help checks ==="
for PROC in "${PROCESSORS[@]}"; do
    echo ""
    echo "────────────────────────────────────────"
    echo "--- ${PROC} ---"
    echo "────────────────────────────────────────"
    if conda run --name "${CONDA_ENV}" "${PROC}" --help; then
        echo ""
        echo "✅ ${PROC} OK"
    else
        echo ""
        echo "❌ ${PROC} FAILED"
    fi
done

echo ""
echo "=== Done ==="

=== Sourcing set_environment.bash ===
✅ Environment activated

=== Processor --help checks ===

────────────────────────────────────────
--- bps_l1_processor ---
────────────────────────────────────────
Usage: bps_l1_processor [OPTIONS] JOB_ORDER_FILE

  BIOMASS Processing Suite - L1 Processor

  Starts processing from Job Order file

Options:
  --working-dir PATH  Working directory (optional)
  --version           Show processor version and exit
  --help              Show this message and exit.


✅ bps_l1_processor OK

────────────────────────────────────────
--- bps_l1_framing_processor ---
────────────────────────────────────────
Usage: bps_l1_framing_processor [OPTIONS] JOB_ORDER_FILE

  BIOMASS Processing Suite - L1 Framing Processor

  Starts processing from Job Order file

Options:
  --working-dir PATH  Working directory (optional)
  --version           Show processor version and exit
  --help              Show this message and exit.


✅ bps_l1_framing_processor OK

────────────